
**Purpose**
----
Download the TEST split from galileo-ai/ragbench, normalize all datasets into a unified schema, and persist them locally for subsequent preprocessing.
---



In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
#Install Dependencies

!pip install -q datasets pandas pyarrow tqdm

In [19]:
#Imports
import os
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

In [29]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/RAGBenchmark"

DATASET_DIR = os.path.join(PROJECT_ROOT, "datasets")
CHUNK_DIR = os.path.join(PROJECT_ROOT, "chunks")
EMBEDDING_DIR = os.path.join(PROJECT_ROOT, "embeddings")
INDEX_DIR = os.path.join(PROJECT_ROOT, "indexes")

for path in [DATASET_DIR, CHUNK_DIR, EMBEDDING_DIR, INDEX_DIR]:
    os.makedirs(path, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATASET_DIR  :", DATASET_DIR)

PROJECT_ROOT : /content/drive/MyDrive/RAGBenchmark
DATASET_DIR  : /content/drive/MyDrive/RAGBenchmark/datasets


In [30]:
#Configuration

RAGBENCH_DATASETS={

    "biomedical":["covidqa","pubmedqa"],
    "general":["hotpotqa","msmarco","hagrid","expertqa"],
    "legal":["cuad"],
    "support":["delucionqa","emanual","techqa"],
    "finance":["finqa","tatqa"]
}

#OUTPUT_DIR = "/content/datasets"
OUTPUT_DIR = DATASET_DIR

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Domains: ",len(RAGBENCH_DATASETS))
print("Datasets:", sum(len(v) for v in RAGBENCH_DATASETS.values()))

Domains:  5
Datasets: 12


In [31]:
print("DATASET_DIR =", DATASET_DIR)
print("OUTPUT_DIR =", OUTPUT_DIR if 'OUTPUT_DIR' in globals() else "Not defined")

DATASET_DIR = /content/drive/MyDrive/RAGBenchmark/datasets
OUTPUT_DIR = /content/drive/MyDrive/RAGBenchmark/datasets


In [32]:
#Schema Normalization Helper
def get_first_available(record, candidate_fields):
  """
  Return the first matching field from the record.
  """
  for field in candidate_fields:
    if field in record and record[field] is not None:
      return record[field]
  return None

In [33]:
#Normalize Dataset

"""Unified Schema:

dataset
domain
id
question
answer
context
metadata
"""
def normalize_dataset(dataset_name, domain, hf_dataset):
    normalized = []

    for idx, row in enumerate(tqdm(hf_dataset, desc=f"Normalizing {dataset_name}")):

        question = get_first_available(
            row,
            [
                "question",
                "query",
                "user_query",
                "input"
            ]
        )

        answer = get_first_available(
            row,
            [
                "answer",
                "answers",
                "gold_answer",
                "reference_answer",
                "output"
            ]
        )

        context = get_first_available(
            row,
            [
                "documents",
                "contexts",
                "context",
                "passages",
                "evidence"
            ]
        )

        normalized.append({
            "dataset": dataset_name,
            "domain": domain,
            "id": row.get("id", idx),
            "question": question,
            "answer": answer,
            "context": context,
            "metadata": row
        })

    return pd.DataFrame(normalized)

In [34]:
#Download and Persist TEST Splits

summary = []
for domain, datasets_list in RAGBENCH_DATASETS.items():
  domain_dir = os.path.join(OUTPUT_DIR, domain)
  os.makedirs(domain_dir, exist_ok=True)

  for dataset_name in datasets_list:
    print("=" * 60)
    print(f"Loading: {dataset_name}")

    try:
      ds = load_dataset( "galileo-ai/ragbench", dataset_name, split="test" )
      print("Rows:", len(ds))

      df = normalize_dataset(
          dataset_name,
          domain, ds
          )

      output_path = os.path.join(
          domain_dir,
          f"{dataset_name}_test.parquet"
          )

      df.to_parquet(output_path, index=False)

      summary.append({
          "domain": domain,
          "dataset": dataset_name,
          "rows": len(df),
          "path": output_path
          })
      print("Saved:", output_path)

    except Exception as e:
      print(f"Failed: {dataset_name}")
      print(e)
print("\nCompleted.")

Loading: covidqa
Rows: 246


Normalizing covidqa:   0%|          | 0/246 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/RAGBenchmark/datasets/biomedical/covidqa_test.parquet
Loading: pubmedqa
Rows: 2450


Normalizing pubmedqa:   0%|          | 0/2450 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/RAGBenchmark/datasets/biomedical/pubmedqa_test.parquet
Loading: hotpotqa
Rows: 390


...

Saved: /content/drive/MyDrive/RAGBenchmark/datasets/support/techqa_test.parquet
Loading: finqa
Rows: 2294


Normalizing finqa:   0%|          | 0/2294 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/RAGBenchmark/datasets/finance/finqa_test.parquet
Loading: tatqa
Rows: 3338


Normalizing tatqa:   0%|          | 0/3338 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/RAGBenchmark/datasets/finance/tatqa_test.parquet

Completed.


In [24]:
#Dataset Summary

summary_df = pd.DataFrame(summary)
summary_df

,domain,dataset,rows,path
0,biomedical,covidqa,246,/content/content/datasets/biomedical/covidqa_t...
1,biomedical,pubmedqa,2450,/content/content/datasets/biomedical/pubmedqa_...
2,general,hotpotqa,390,/content/content/datasets/general/hotpotqa_tes...
3,general,msmarco,423,/content/content/datasets/general/msmarco_test...
4,general,hagrid,1318,/content/content/datasets/general/hagrid_test....
5,general,expertqa,203,/content/content/datasets/general/expertqa_tes...
6,legal,cuad,510,/content/content/datasets/legal/cuad_test.parquet
7,support,delucionqa,184,/content/content/datasets/support/delucionqa_t...
8,support,emanual,132,/content/content/datasets/support/emanual_test...
9,support,techqa,314,/content/content/datasets/support/techqa_test....


In [25]:
#Persist Summary

summary_path = os.path.join(
    OUTPUT_DIR,
    "dataset_summary.csv"
    )
summary_df.to_csv(summary_path, index=False)
print("Summary saved:", summary_path)

Summary saved: /content/content/datasets/dataset_summary.csv


In [26]:
#Validate Saved Files

for root, dirs, files in os.walk(OUTPUT_DIR):
  level = root.replace(OUTPUT_DIR, "").count(os.sep)
  indent = ""*4*level

  print(f"{indent}{os.path.basename(root)}/")

  for file in files:
    print(f"{indent}  {file}")

datasets/
  dataset_summary.csv
finance/
  tatqa_test.parquet
  finqa_test.parquet
legal/
  cuad_test.parquet
biomedical/
  pubmedqa_test.parquet
  covidqa_test.parquet
support/
  delucionqa_test.parquet
  emanual_test.parquet
  techqa_test.parquet
general/
  msmarco_test.parquet
  hagrid_test.parquet
  hotpotqa_test.parquet
  expertqa_test.parquet


In [27]:
#Optional : Download Assets

from google.colab import files

!zip -rq datasets.zip /content/datasets

files.download("datasets.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>